# BER na saída do equalizador: modelo analítico para sistemas IMDD M-PAM

Este notebook organiza a etapa final do modelo analítico apresentado no artigo *An Analytical Model for Performance Estimation in Modern High-Capacity IMDD Systems*.

O objetivo aqui é calcular a **BER estimada na saída de equalizadores FFE e DFE**, assumindo que etapas anteriores do modelo já foram calculadas, principalmente:

- a SNR espectral dobrada, isto é, $\overline{SNR}(f)$;
- o eixo de frequências da banda de integração;
- o período de símbolo $T$;
- a ordem da modulação PAM, $M$.

Fluxo implementado:

$$\overline{SNR}(f) \longrightarrow SNR_{FFE} \text{ ou } SNR_{DFE} \longrightarrow BER$$


## 1. Ideia teórica

No modelo analítico, primeiro calcula-se uma SNR dependente da frequência. Após considerar o dobramento espectral causado pela amostragem, obtém-se:

$$\overline{SNR}(f)$$

Essa quantidade representa a SNR espectral efetiva dentro da banda de Nyquist do equalizador:

$$-\frac{1}{2T} \leq f \leq \frac{1}{2T}$$

A partir dela, o artigo calcula a SNR na saída do equalizador. As expressões são diferentes para FFE e DFE porque esses equalizadores tratam a interferência intersimbólica de maneira diferente.


## 2. SNR na saída do FFE

Para o equalizador feed-forward, a SNR efetiva é:

$$SNR_{FFE} = \frac{1}{T}\left[\int_{-\frac{1}{2T}}^{\frac{1}{2T}} \frac{1}{\overline{SNR}(f)+1}\,df\right]^{-1} - 1$$

Essa fórmula penaliza mais fortemente regiões de baixa SNR. Isso faz sentido porque o FFE precisa compensar atenuações do canal e, ao fazer isso, pode amplificar ruído em frequências degradadas.


## 3. SNR na saída do DFE

Para o decision-feedback equalizer, a SNR efetiva é:

$$SNR_{DFE} = \exp\left[T\int_{-\frac{1}{2T}}^{\frac{1}{2T}} \log\left(\overline{SNR}(f)+1\right)\,df\right] - 1$$

O DFE costuma apresentar desempenho melhor que o FFE em canais com forte limitação de banda, pois consegue combater parte da interferência intersimbólica usando realimentação por decisão.


## 4. Conversão de SNR para BER em M-PAM

Depois de obter a SNR efetiva na saída do equalizador, a BER aproximada para M-PAM em ruído gaussiano aditivo é:

$$BER \approx \frac{M-1}{M\log_2(M)}\operatorname{erfc}\left(\sqrt{\frac{3\,SNR}{2(M^2-1)}}\right)$$

Aqui, a SNR deve estar em escala linear, não em dB. Para 4-PAM, usa-se $M=4$.


In [ ]:
import numpy as np
from scipy.special import erfc
import matplotlib.pyplot as plt


In [ ]:
def snr_equalizer_output_ffe(snr_folded, freqs_base, T):
    """
    Calcula a SNR efetiva na saída de um equalizador FFE.

    Parâmetros
    ----------
    snr_folded : float ou np.ndarray
        SNR espectral dobrada, isto é, SNR_bar(f), em escala linear.

    freqs_base : np.ndarray
        Eixo de frequências dentro da banda de Nyquist:
        [-1/(2T), 1/(2T)].

    T : float
        Período de símbolo, em segundos.

    Retorna
    -------
    snr_ffe : float
        SNR efetiva na saída do FFE, em escala linear.
    """
    snr_folded = np.asarray(snr_folded, dtype=float)
    freqs_base = np.asarray(freqs_base, dtype=float)

    if np.any(snr_folded < 0):
        raise ValueError("snr_folded deve ser não negativa.")

    integrand = 1 / (snr_folded + 1)
    integral = np.trapz(integrand, freqs_base)

    return (1 / T) * (1 / integral) - 1


def snr_equalizer_output_dfe(snr_folded, freqs_base, T):
    """
    Calcula a SNR efetiva na saída de um equalizador DFE.
    """
    snr_folded = np.asarray(snr_folded, dtype=float)
    freqs_base = np.asarray(freqs_base, dtype=float)

    if np.any(snr_folded < 0):
        raise ValueError("snr_folded deve ser não negativa.")

    integrand = np.log(snr_folded + 1)
    integral = np.trapz(integrand, freqs_base)

    return np.exp(T * integral) - 1


def ber_from_snr_m_pam(snr_linear, M):
    """
    Calcula a BER aproximada para M-PAM a partir da SNR em escala linear.

    Fórmula:
        BER ≈ (M - 1)/(M log2(M)) * erfc(
            sqrt(3*SNR / (2*(M^2 - 1)))
        )
    """
    snr_linear = np.asarray(snr_linear, dtype=float)

    if np.any(snr_linear < 0):
        raise ValueError("snr_linear deve ser não negativa.")

    return ((M - 1) / (M * np.log2(M))) * erfc(
        np.sqrt((3 * snr_linear) / (2 * (M**2 - 1)))
    )


def linear_to_db(x):
    return 10 * np.log10(x)


def db_to_linear(x_db):
    return 10**(x_db / 10)


## 5. Função única para obter BER de FFE e DFE

A função abaixo recebe diretamente a SNR dobrada calculada anteriormente pelo grupo e retorna as SNRs e BERs correspondentes para FFE e DFE.


In [ ]:
def ber_equalizer_outputs_from_folded_snr(snr_folded, freqs_base, T, M):
    """
    Calcula SNR e BER analíticas para FFE e DFE a partir de SNR_bar(f).
    """
    snr_ffe = snr_equalizer_output_ffe(snr_folded, freqs_base, T)
    snr_dfe = snr_equalizer_output_dfe(snr_folded, freqs_base, T)

    ber_ffe = ber_from_snr_m_pam(snr_ffe, M)
    ber_dfe = ber_from_snr_m_pam(snr_dfe, M)

    return {
        "snr_ffe_linear": snr_ffe,
        "snr_ffe_db": linear_to_db(snr_ffe),
        "ber_ffe": ber_ffe,
        "snr_dfe_linear": snr_dfe,
        "snr_dfe_db": linear_to_db(snr_dfe),
        "ber_dfe": ber_dfe,
    }


## 6. Exemplo artificial para testar o fluxo

Nesta seção é criado um exemplo simples apenas para verificar se as funções estão operando corretamente. No uso real, substitua `snr_folded_example` pela SNR dobrada calculada pelo grupo.


In [ ]:
Rs = 25e9          # 25 GBaud
T = 1 / Rs         # 40 ps
M = 4              # 4-PAM

Nfreq = 4096
freqs_base = np.linspace(-Rs/2, Rs/2, Nfreq)

# Exemplo artificial: SNR dobrada levemente menor nas bordas da banda
snr_center_db = 18
snr_edge_db = 12

x = np.abs(freqs_base) / (Rs/2)
snr_profile_db = snr_center_db - (snr_center_db - snr_edge_db) * x**2
snr_folded_example = db_to_linear(snr_profile_db)

results = ber_equalizer_outputs_from_folded_snr(
    snr_folded=snr_folded_example,
    freqs_base=freqs_base,
    T=T,
    M=M
)

for key, value in results.items():
    print(f"{key}: {value}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(freqs_base / 1e9, snr_profile_db)
plt.xlabel("Frequência [GHz]")
plt.ylabel(r"$\overline{SNR}(f)$ [dB]")
plt.title("Exemplo de SNR dobrada dentro da banda de integração")
plt.grid(True)
plt.show()


## 7. Curva BER versus SNR para 4-PAM

A célula abaixo mostra a curva teórica de BER em função da SNR para 4-PAM. Essa visualização ajuda a verificar se os valores obtidos estão em uma faixa física razoável.


In [ ]:
snr_db_axis = np.linspace(0, 30, 301)
snr_linear_axis = db_to_linear(snr_db_axis)

ber_axis = ber_from_snr_m_pam(snr_linear_axis, M=4)

plt.figure(figsize=(7, 4))
plt.semilogy(snr_db_axis, ber_axis)
plt.xlabel("SNR [dB]")
plt.ylabel("BER")
plt.title("BER teórica para 4-PAM em função da SNR")
plt.grid(True, which="both")
plt.ylim(1e-8, 1)
plt.show()


## 8. Como integrar com o código do grupo

Quando seus colegas já tiverem calculado `snr_folded`, `freqs_base` e `T`, você só precisa executar:

```python
results = ber_equalizer_outputs_from_folded_snr(
    snr_folded=snr_folded,
    freqs_base=freqs_base,
    T=T,
    M=4
)

print("FFE")
print("SNR dB:", results["snr_ffe_db"])
print("BER:", results["ber_ffe"])

print("DFE")
print("SNR dB:", results["snr_dfe_db"])
print("BER:", results["ber_dfe"])
```

Atenção: `snr_folded` deve estar em escala linear. Se estiver em dB, converta com:

```python
snr_folded = db_to_linear(snr_folded_db)
```


## 9. Observações importantes

1. A SNR usada na fórmula de BER deve ser a SNR efetiva na saída do equalizador, isto é, `snr_ffe` ou `snr_dfe`, não diretamente a SNR espectral original.

2. A fórmula de BER assume uma aproximação gaussiana para o ruído.

3. Para ruídos dependentes do nível de sinal, como RIN e shot noise, o artigo comenta uma extensão heurística: calcular BER por olho interno do diagrama PAM e depois tirar a média. A função simples deste notebook calcula a BER diretamente a partir de uma única SNR efetiva.

4. Se a BER aparecer como `0.0`, isso pode significar que a SNR está muito alta e a função `erfc` sofreu underflow numérico. Nesse caso, imprima também a SNR em dB para verificar a escala.
